In [1]:
import requests
import pandas as pd

In [ ]:
!pip install gtfs-realtime-bindings pandas requests

from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict
import pandas as pd
from requests import get

# Sample GTFS-R URL from Malaysia's Open API
URL = 'https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=rapid-bus-kl'

# Parse the GTFS Realtime feed
feed = gtfs_realtime_pb2.FeedMessage()
response = get(URL)
feed.ParseFromString(response.content)

# Extract and print vehicle position information
vehicle_positions = [MessageToDict(entity.vehicle) for entity in feed.entity]
print(f'Total vehicles: {len(vehicle_positions)}')
df = pd.json_normalize(vehicle_positions)
print(df)

In [10]:
import requests
import zipfile
import io
import pandas as pd

def fetch_gtfs_data(category):
    # Download the GTFS ZIP file from the API
    url = f"https://api.data.gov.my/gtfs-static/prasarana?category={category}"
    response = requests.get(url)

    if response.status_code != 200:
        print(response.status_code)
        raise Exception(f"Failed to fetch GTFS data for category '{category}'")

    # Extract ZIP file directly into memory
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        # Read relevant files into DataFrames
        gtfs_data = {}
        files = ['stops.txt', 'routes.txt', 'trips.txt', 'shapes.txt']

        for file in files:
            if file in zip_ref.namelist():
                with zip_ref.open(file) as f:
                    gtfs_data[file.replace('.txt', '')] = pd.read_csv(f)

        return gtfs_data

rapidkl_rail_kl_gtfs = fetch_gtfs_data('rapid-rail-kl')

# Access the data
# agency = rapidkl_rail_kl_gtfs.get('agency')
stops = rapidkl_rail_kl_gtfs.get('stops')
routes = rapidkl_rail_kl_gtfs.get('routes')
trips = rapidkl_rail_kl_gtfs.get('trips')
shapes = rapidkl_rail_kl_gtfs.get('shapes')

In [19]:
stops[['category','route_id']].value_counts().sort_index()

category  route_id
BRT       BRT          7
LRT       AG          18
          KJ          37
          PH          29
MR        MR          11
MRT       MRT         29
          PYL         36
Name: count, dtype: int64

In [31]:
stops = stops[stops['route_id'].isin(['AG','MR'])]
stops

,stop_id,stop_name,stop_lat,stop_lon,category,route_id,geometry,isOKU,status,search
0,AG18,AMPANG,3.150318,101.760049,LRT,AG,[object Object],True,valid,LRT AMPANG
1,AG17,CAHAYA,3.140575,101.756677,LRT,AG,[object Object],True,valid,LRT CAHAYA
2,AG16,CEMPAKA,3.138324,101.752979,LRT,AG,[object Object],True,valid,LRT CEMPAKA
3,AG15,PANDAN INDAH,3.134581,101.746509,LRT,AG,[object Object],True,valid,LRT PANDAN INDAH
4,AG14,PANDAN JAYA,3.130141,101.739122,LRT,AG,[object Object],True,valid,LRT PANDAN JAYA
5,AG13,MALURI,3.123290,101.727283,LRT,AG,[object Object],True,valid,LRT MALURI
6,AG12,MIHARJA,3.120973,101.717922,LRT,AG,[object Object],True,valid,LRT MIHARJA
7,AG11,CHAN SOW LIN,3.128105,101.715637,LRT,AG,[object Object],True,valid,LRT CHAN SOW LIN
8,AG10,PUDU,3.134879,101.711957,LRT,AG,[object Object],True,valid,LRT PUDU
9,AG9,HANG TUAH,3.140012,101.705984,LRT,AG,[object Object],True,valid,LRT HANG TUAH


Getting the starting point for the pedestrian.
Heuristics:
1. Should be within 1km walking distance from original location.

In [24]:
orig = (3.1574593436501144, 101.710740684479)  # KLCC